<a href="https://colab.research.google.com/github/kavyashrenivuppugandla-cmd/ASSIGNMENT-NLP-5-Fine-tune-a-transformer-model-/blob/main/ASSIGNMENT_NLP_%E2%80%93_5_Fine_tune_a_transformer_model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os

from google.colab import files
uploaded = files.upload()

file_name = [f for f in os.listdir() if f.endswith(".ipynb")][0]
print("Found file:", file_name)

with open(file_name, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open("fixed_notebook.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("Fixed notebook ready")

Saving ASSIGNMENT_NLP_–_5_Fine_tune_a_transformer_model_.ipynb to ASSIGNMENT_NLP_–_5_Fine_tune_a_transformer_model_.ipynb
Found file: ASSIGNMENT_NLP_–_5_Fine_tune_a_transformer_model_.ipynb
Fixed notebook ready


In [ ]:

# 1. Install Required Libraries

!pip uninstall -y transformers accelerate peft datasets -q
!pip install transformers==4.35.2 datasets==2.14.6 accelerate==0.25.0 seqeval torch conllu tabulate pandas -q

# Disable wandb
import os
os.environ["WANDB_DISABLED"] = "true"



# 2. Import Libraries
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
import pandas as pd
from tabulate import tabulate

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# 3. Load Dataset

dataset = load_dataset("universal_dependencies", "en_ewt")

print("\nDATASET INFO")
print(dataset)

# 4. Labels

label_list = dataset["train"].features["upos"].feature.names

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# 5. Load DistilBERT Tokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer.model_max_length = 128

# 6. Tokenization

def tokenize_and_align_labels(example):

    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["upos"][word_idx])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_datasets = dataset.map(tokenize_and_align_labels)

# 7. Load DistilBERT Model
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("\nModel Loaded: DistilBERT")

# 8. Training Config
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="no"
)

# 9. Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# 10. Metrics

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []

        for p, l in zip(pred, lab):
            if l != -100:
                temp_pred.append(id2label[p])
                temp_lab.append(id2label[l])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

# 11. Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(1500)),
    eval_dataset=tokenized_datasets["validation"].select(range(300)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 12. Train

print("\n TRAINING STARTED ")
trainer.train()
print("\n TRAINING COMPLETED ")

# 13. Evaluation

results = trainer.evaluate()

print("\n RESULTS ")

print(tabulate(df, headers="keys", tablefmt="grid"))

# 15. Inference

import torch

sentence = "John works at Google in California"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer(
    sentence.split(),
    return_tensors="pt",
    is_split_into_words=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)

predictions = np.argmax(outputs.logits.detach().cpu().numpy(), axis=2)

word_ids = tokenizer(sentence.split(), is_split_into_words=True).word_ids()

predicted_labels = []

for idx, word_idx in enumerate(word_ids):
    if word_idx is not None:
        predicted_labels.append(id2label[predictions[0][idx]])

print("\n INFERENCE ")
print("Sentence:", sentence)
print("POS Tags:", predicted_labels)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.

DATASET INFO
DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2077
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,2.069100,0.535318,0.864264,0.858176,0.861209


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,2.069100,0.535318,0.864264,0.858176,0.861209
2,0.405100,0.361522,0.899644,0.890805,0.895203



 TRAINING COMPLETED 



 RESULTS 
+----+----------+-------------+----------+---------+
|    |     Loss |   Precision |   Recall |      F1 |
+====+==========+=============+==========+=========+
|  0 | 0.389168 |    0.890449 | 0.881535 | 0.88597 |
+----+----------+-------------+----------+---------+

 INFERENCE 
Sentence: John works at Google in California
POS Tags: ['PROPN', 'VERB', 'ADP', 'PROPN', 'ADP', 'PROPN']
